# Clase 6 — Repaso integrador para la primera entrega

**Objetivo:** unir todo lo visto (AMq1 + clases 1 a 5) en una sola foto, practicar con un examen
mezclado y preparar la defensa oral del TP.

Cómo usar esta clase:

1. Leé el **mapa** y el **recorrido de un día del pipeline** (§1 y §2): si algo no te cierra, volvé a
   la clase correspondiente.
2. Resolvé el ejercicio integrador ✏️ (§4).
3. Hacé el **examen** (§5): cada vez que ejecutás la celda salen 20 preguntas distintas de las 7 clases.
4. Pasá las **tarjetas** (§6) hasta saberlas todas.
5. Respondé en voz alta las **preguntas de defensa** (§7) antes de abrir la respuesta.

In [1]:
from helpers import ROOT, check, cuestionario, tarjetas

import ipywidgets as widgets
from IPython.display import Markdown, display

## 1. 📖 El mapa completo

```mermaid
flowchart TD
    CSV[("CSV en data/raw<br>50.000 estudiantes · R1")] -- "DAG etl_burnout · C4" --> MINIO[("MinIO s3://data/burnout<br>raw · train · test")]
    MINIO -- "notebook de Optuna · R2" --> EXP["MLflow: experimento Burnout Risk<br>40 runs hijos · C3"]
    EXP -- "mejores hiperparámetros" --> TRN["DAG train_burnout · C4"]
    MINIO --> TRN
    TRN -- "registra y pone el alias" --> REG["MLflow Registry<br>burnout_model_prod@champion · C3"]
    REG --> BAT["DAG batch_predict_burnout<br>@daily · C4 y C5"]
    MINIO --> BAT
    BAT --> PRED[("predictions.csv · C5")]
```

Todo eso corre dentro del **docker-compose** (C2), y todas las tareas importan el mismo paquete
`src/burnout`, con tests y linter (C1).

| Clase | Pregunta que responde | Pieza del TP |
|---|---|---|
| R1 | ¿qué datos puedo usar y cómo los preparo? | `config.py`, `data.py` |
| R2 | ¿qué modelo y con qué hiperparámetros? | `model.py`, `BEST_PARAMS_AMQ1` |
| C1 | ¿cómo hago que el código sea mantenible y reproducible? | paquete, `tests/`, `ruff`, Git |
| C2 | ¿cómo corre igual en cualquier máquina? | `docker-compose.yaml`, `dockerfiles/` |
| C3 | ¿qué probé, qué funcionó, qué está en producción? | experimento, Registry, alias |
| C4 | ¿quién corre el pipeline y cuándo? | `airflow/dags/` |
| C5 | ¿cómo llegan las predicciones a quien decide? | DAG batch, `predictions.csv` |

## 2. 🏭 Caso integrador: un día en la vida del pipeline

Seguimos al sistema durante un ciclo completo. Para cada paso: qué pasa, quién lo hace y qué concepto
lo sostiene.

| # | Momento | Qué pasa | Dónde | Concepto |
|---|---|---|---|---|
| 1 | Una vez | `docker compose --profile all up -d` levanta Airflow, MLflow, MinIO, PostgreSQL, Valkey y FastAPI | Docker | imágenes, volúmenes, redes (C2) |
| 2 | Al disparar el ETL | `extract` sube el CSV a `s3://data/burnout/raw/` | Airflow → MinIO | DAG, idempotencia (C4) |
| 3 | | `validate_raw` revisa columnas, nulos y categorías | `burnout.data.validate` | fallar temprano (R1, C1) |
| 4 | | `split` genera train/test 80/20 estratificado | `burnout.etl.split` | fuga de información, estratificación (R1) |
| 5 | | `register_dataset` guarda `data.json` y un run con `log_input` | MLflow | trazabilidad de datos (C3) |
| 6 | Experimentación | el notebook corre 40 pruebas de Optuna, cada una como run hijo | MLflow | runs anidados, CV (R2, C3) |
| 7 | Al disparar el entrenamiento | `train_burnout` busca los mejores parámetros con `search_runs` | MLflow | reproducibilidad (C1) |
| 8 | | entrena, evalúa en test y registra la versión nueva con alias `champion` | Registry | alias, promoción (C3) |
| 9 | Todos los días a las 00:00 | `batch_predict_burnout` toma un lote, carga `@champion` y predice | Airflow + MLflow | batch, cron (C4, C5) |
| 10 | A la mañana | la coordinación lee `predictions.csv` y contacta a los de riesgo alto | MinIO | servir predicciones precalculadas (C5) |
| 11 | Si algo sale mal | se vuelve el alias `champion` a la versión anterior | Registry | rollback (C3, C5) |

**¿Y si falla algo?**

- *El CSV llega con una categoría nueva* → falla el paso 3, el DAG queda en rojo y no se entrena con
  datos rotos.
- *MinIO tarda en responder* → Airflow reintenta la tarea, y como es idempotente no ensucia nada.
- *El modelo nuevo es peor* → nadie cambia código: se mueve el alias.

## 3. 📖 Glosario

| Término | Definición corta | Clase |
|---|---|---|
| Fuga de información | usar datos que no existirán al momento de predecir | R1 |
| Split estratificado | train/test con las mismas proporciones de clases | R1 |
| F1 macro | promedio simple del F1 de cada clase | R2 |
| Validación cruzada | estimar el rendimiento rotando particiones de train | R2 |
| TPE | muestreador de Optuna que aprende de las pruebas anteriores | R2 |
| MLOps | prácticas para llevar y mantener modelos en producción | C1 |
| DRY | cada regla en un solo lugar | C1 |
| Reproducibilidad | mismo código + datos + entorno = mismo modelo | C1 |
| Imagen / contenedor | plantilla inmutable / instancia que corre | C2 |
| Volumen | almacenamiento que sobrevive al contenedor | C2 |
| Run / experimento | una ejecución registrada / grupo de runs | C3 |
| Model Registry | catálogo versionado de modelos | C3 |
| Alias `champion` | etiqueta móvil que apunta a la versión en producción | C3 |
| DAG | grafo de tareas con dependencias y sin ciclos | C4 |
| XCom | valor chico que pasa de una tarea a otra | C4 |
| Idempotencia | correr dos veces = correr una | C4 |
| Batch / online | predecir en lote programado / al recibir cada pedido | C5 |
| Canary / shadow | parte del tráfico al modelo nuevo / copia del tráfico sin responder | C5 |

## 4. ✏️ Ejercicio integrador: ¿qué servicio se encarga?

Completá el diccionario con la **herramienta** responsable de cada necesidad (en el compose, MinIO es el
servicio `s3` y Valkey el servicio `redis`).
Opciones: `"airflow"`, `"mlflow"`, `"minio"`, `"postgres"`, `"valkey"`, `"fastapi"`.

In [2]:
servicio_para = {
    "guardar los CSV de train y test": "",
    "registrar parámetros y métricas de cada prueba de Optuna": "",
    "programar el ETL y la predicción diaria": "",
    "guardar los metadatos de Airflow y de MLflow (runs, DAG runs)": "",
    "hacer de cola (broker) entre el scheduler y los workers de Celery": "",
    "responder predicciones online por HTTP (2ª entrega)": "",
}  # ← completá cada valor

In [3]:
esperado = {
    "guardar los CSV de train y test": "minio",
    "registrar parámetros y métricas de cada prueba de Optuna": "mlflow",
    "programar el ETL y la predicción diaria": "airflow",
    "guardar los metadatos de Airflow y de MLflow (runs, DAG runs)": "postgres",
    "hacer de cola (broker) entre el scheduler y los workers de Celery": "valkey",
    "responder predicciones online por HTTP (2ª entrega)": "fastapi",
}
mal = [k for k in esperado if servicio_para.get(k, "").strip().lower() != esperado[k]]
check(
    "Ejercicio integrador",
    not mal,
    f"revisá: {mal[0]!r}. Pensá si es un archivo (MinIO), un metadato (PostgreSQL) o una cola (Valkey)."
    if mal else "",
)

⏳ **Ejercicio integrador**: todavía no. *Pista:* revisá: 'guardar los CSV de train y test'. Pensá si es un archivo (MinIO), un metadato (PostgreSQL) o una cola (Valkey).

False

<details><summary>Ver solución</summary>

```python
servicio_para = {
    "guardar los CSV de train y test": "minio",
    "registrar parámetros y métricas de cada prueba de Optuna": "mlflow",
    "programar el ETL y la predicción diaria": "airflow",
    "guardar los metadatos de Airflow y de MLflow (runs, DAG runs)": "postgres",
    "hacer de cola (broker) entre el scheduler y los workers de Celery": "valkey",
    "responder predicciones online por HTTP (2ª entrega)": "fastapi",
}
```

Ojo con la segunda: MLflow **recibe** las métricas, pero las guarda en PostgreSQL (metadatos) y los
modelos en MinIO (artefactos).
</details>

## 5. 📝 Examen integrador

Cada ejecución arma **20 preguntas al azar** de las 7 clases (hay 56 en total). Ejecutala varias veces
hasta sacar más del 90 % seguido.

In [4]:
cuestionario(["R1", "R2", "C1", "C2", "C3", "C4", "C5"], n=20, mezclar=True)

### Examen por tema
Elegí las clases que querés repasar y apretá **Generar**.

In [5]:
from banco import NOMBRES

temas = widgets.SelectMultiple(options=[(v, k) for k, v in NOMBRES.items()], value=["C3", "C4"], rows=7)
cantidad = widgets.IntSlider(value=10, min=5, max=30, description="Preguntas")
generar = widgets.Button(description="Generar", button_style="primary")
salida = widgets.Output()


def on_generar(_):
    salida.clear_output()
    with salida:
        cuestionario(list(temas.value), n=cantidad.value, mezclar=True)


generar.on_click(on_generar)
display(widgets.HBox([temas, widgets.VBox([cantidad, generar])]), salida)

Output()

## 6. 🔁 Tarjetas de todo el curso
63 tarjetas. Apretá **Mezclar** para recorrerlas en otro orden.

In [6]:
tarjetas(["R1", "R2", "C1", "C2", "C3", "C4", "C5"])

## 7. 🎤 Preguntas de defensa oral

Respondé en voz alta (o escribiendo) **antes** de abrir la respuesta modelo. Son las que suelen
aparecer cuando se presenta un TP de MLOps.

<details><summary><b>1. ¿Por qué no alcanza con el notebook de AMq1 para "estar en producción"?</b></summary>

Porque se corre a mano, no es reproducible garantizadamente, no registra qué se probó, no tiene
tests, y nadie lo ejecuta solo cuando llegan datos nuevos. El TP lo convierte en un paquete testeado,
orquestado por Airflow, con experimentos y modelos versionados en MLflow y todo corriendo en Docker.
</details>

<details><summary><b>2. ¿Cómo garantizan que el modelo sea reproducible?</b></summary>

Semilla fija (`SEED = 42`) en split, CV, Optuna y XGBoost; hilos fijos (`XGB_N_JOBS = 4`); versiones
exactas en `requirements.txt` y las **mismas** en las imágenes de Docker; datos versionados con su
SHA-256. Un test (`test_entrenamiento_determinista`) lo vigila.
</details>

<details><summary><b>3. ¿Por qué el F1 no da exactamente el 0,5405 de AMq1?</b></summary>

Porque XGBoost con `tree_method="hist"` cambia levemente según la cantidad de hilos (el orden de las
sumas de punto flotante), y en AMq1 se usó `n_jobs=-1`. Con 4 hilos da 0,5365, dentro del IC 95 % de
AMq1 (0,531 a 0,550). Preferimos un resultado igual en cualquier máquina a uno que dependa de la
cantidad de núcleos.
</details>

<details><summary><b>4. ¿Qué features excluyeron y por qué?</b></summary>

`Post_Semester_GPA` y `Skill_Retention_Score`: se miden al final del semestre, junto con el burnout.
Al momento de predecir no existen, así que usarlas sería fuga de información.
</details>

<details><summary><b>5. ¿Cómo pasan los datos de una tarea de Airflow a otra?</b></summary>

Los datos (CSV) se escriben en MinIO (`s3://data/burnout/...`). Por XCom solo viajan rutas y
resúmenes chicos. Así las tareas son independientes, idempotentes y la base de Airflow no crece.
</details>

<details><summary><b>6. ¿Cómo se pone un modelo nuevo en producción? ¿Y cómo se vuelve atrás?</b></summary>

Se registra como versión nueva de `burnout_model_prod` y se mueve el alias `champion` a esa versión.
El DAG batch siempre carga `models:/burnout_model_prod@champion`, así que la próxima corrida usa el
nuevo sin tocar código. Para volver atrás, se mueve el alias a la versión anterior.
</details>

<details><summary><b>7. ¿Por qué batch y no una API para esta primera entrega?</b></summary>

El caso de uso (la facultad revisa periódicamente quién está en riesgo) no necesita respuesta
inmediata, y batch es más simple y barato. La API online con FastAPI es parte de la 2ª entrega, para
casos en los que la predicción se necesita al instante.
</details>

<details><summary><b>8. ¿Qué pasa si llega un CSV con un error?</b></summary>

`validate()` levanta `ValueError` en la tarea `validate_raw`, el DAG queda en rojo y no se genera un
split ni se entrena con datos rotos. Se ve en la UI de Airflow con el mensaje exacto.
</details>

<details><summary><b>9. ¿Cómo prueban la lógica de los DAGs sin levantar Airflow?</b></summary>

Los DAGs son envoltorios finos de funciones de `burnout.etl` y `burnout.batch`. Los tests las corren
con un "S3 falso" (se reemplaza `storage._client` con `monkeypatch`) y MLflow con SQLite. `pytest`
tarda unos 10 segundos.
</details>

<details><summary><b>10. ¿Qué agregarían para llegar al nivel 2 de madurez de MLOps?</b></summary>

CI/CD: que cada Pull Request corra `ruff` y `pytest` automáticamente (por ejemplo con GitHub Actions),
que se construyan y publiquen las imágenes, y un reentrenamiento automático con comparación
champion/challenger y monitoreo del *drift* de los datos.
</details>

## 8. ✅ Checklist de la primera entrega
Marcalo a medida que lo verifiques. El detalle está en [docs/entrega_1.md](../docs/entrega_1.md).

In [7]:
items = [
    "Repo público con los 3 integrantes y README",
    "Modelo de AMq1 empaquetado en src/burnout con tests que pasan",
    "docker compose --profile all up -d levanta todos los servicios healthy",
    "DAG etl_burnout corre en verde y deja los datos en MinIO",
    "Experimento de Optuna registrado en MLflow (40 runs hijos)",
    "DAG train_burnout registra burnout_model_prod con alias champion",
    "DAG batch_predict_burnout escribe predictions.csv",
    "Capturas de Airflow, MLflow y MinIO en docs/",
    "Cada integrante puede explicar las 10 preguntas de defensa",
]
cajas = [widgets.Checkbox(description=t, indent=False, layout=widgets.Layout(width="auto")) for t in items]
avance = widgets.HTML()


def actualizar(_=None):
    hechas = sum(c.value for c in cajas)
    avance.value = f"<b>{hechas} / {len(cajas)}</b> {'🎉 ¡Listos para entregar!' if hechas == len(cajas) else ''}"


for c in cajas:
    c.observe(actualizar, names="value")
actualizar()
display(widgets.VBox([*cajas, avance]))

## Resumen
- El TP es un ciclo: datos → validación → split → experimentos → registro → predicción batch.
- Cada herramienta resuelve una pregunta: Docker (¿corre igual?), MLflow (¿qué funcionó y qué está en
  producción?), Airflow (¿quién lo corre y cuándo?).
- La reproducibilidad y la trazabilidad son lo que separa un notebook de un sistema en producción.

¡Éxitos en la entrega! 🚀